# 07 - Embedding Analysis

t-SNE visualization and cosine similarity analysis of speaker embeddings.

In [ ]:
import sys
sys.path.insert(0, '..')

import pickle
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader

from src.config import load_config, FeatureConfig, AudioConfig
from src.data.download import prepare_dataset
from src.data.splits import load_splits, get_split_metadata
from src.data.dataset import SpeakerDataset
from src.models.cnn import SpeakerCNN
from src.models.ecapa_tdnn import ECAPATDNN
from src.evaluation.embeddings import extract_embeddings
from src.evaluation.visualization import plot_tsne
from src.utils import set_seed, get_device

set_seed(42)
device = get_device('auto')
plt.style.use('seaborn-v0_8-paper')

In [ ]:
# Load test data
config = load_config('../configs/base.yaml')
metadata = prepare_dataset(config.data.data_dir, config.data.num_speakers, config.data.min_utterances_per_speaker)
splits = load_splits(config.data.data_dir)
test_meta = get_split_metadata(metadata, splits, 'test')

le_path = Path(config.data.data_dir) / 'splits' / 'label_encoder.pkl'
with open(le_path, 'rb') as f:
    label_encoder = pickle.load(f)

test_dataset = SpeakerDataset(
    test_meta, FeatureConfig(feature_type='mel_spectrogram'), AudioConfig(),
    label_encoder=label_encoder, train=False,
)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [ ]:
# Extract CNN embeddings
cnn_ckpt = Path('../checkpoints/cnn_mel_spectrogram_best.pt')
if cnn_ckpt.exists():
    ckpt = torch.load(cnn_ckpt, map_location=device, weights_only=False)
    cnn_model = SpeakerCNN(
        num_speakers=ckpt['config']['num_speakers'],
        embedding_dim=ckpt['config']['embedding_dim'],
    )
    cnn_model.load_state_dict(ckpt['model_state_dict'])
    cnn_model = cnn_model.to(device)
    
    cnn_emb, cnn_labels = extract_embeddings(cnn_model, test_loader, device)
    print(f'CNN embeddings: {cnn_emb.shape}')
    
    plot_tsne(cnn_emb, cnn_labels, 
              speaker_names=list(label_encoder.classes_),
              save_path='../results/tsne_cnn.png')
else:
    print('CNN checkpoint not found. Train the model first.')

In [ ]:
# Extract ECAPA-TDNN embeddings
ecapa_ckpt = Path('../checkpoints/ecapa_tdnn_best.pt')
if ecapa_ckpt.exists():
    ckpt = torch.load(ecapa_ckpt, map_location=device, weights_only=False)
    ecapa_model = ECAPATDNN(
        num_speakers=ckpt['config']['num_speakers'],
        embedding_dim=ckpt['config']['embedding_dim'],
        channels=512, scale=8,
    )
    ecapa_model.load_state_dict(ckpt['model_state_dict'])
    ecapa_model = ecapa_model.to(device)
    
    ecapa_emb, ecapa_labels = extract_embeddings(ecapa_model, test_loader, device)
    print(f'ECAPA-TDNN embeddings: {ecapa_emb.shape}')
    
    plot_tsne(ecapa_emb, ecapa_labels,
              speaker_names=list(label_encoder.classes_),
              save_path='../results/tsne_ecapa.png')
else:
    print('ECAPA-TDNN checkpoint not found. Train the model first.')

In [ ]:
# Cosine similarity distribution
def compute_cosine_similarity_distributions(embeddings, labels):
    """Compute intra-speaker and inter-speaker cosine similarity distributions."""
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-8
    normed = embeddings / norms
    
    intra_scores = []
    inter_scores = []
    
    n = min(len(embeddings), 500)  # Subsample for speed
    indices = np.random.choice(len(embeddings), n, replace=False)
    
    for i in range(n):
        for j in range(i + 1, n):
            ii, jj = indices[i], indices[j]
            score = float(np.dot(normed[ii], normed[jj]))
            if labels[ii] == labels[jj]:
                intra_scores.append(score)
            else:
                inter_scores.append(score)
    
    return intra_scores, inter_scores

if 'ecapa_emb' in dir():
    intra, inter = compute_cosine_similarity_distributions(ecapa_emb, ecapa_labels)
    
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.hist(inter, bins=50, alpha=0.6, label=f'Inter-speaker (n={len(inter)})', color='red', density=True)
    ax.hist(intra, bins=50, alpha=0.6, label=f'Intra-speaker (n={len(intra)})', color='blue', density=True)
    ax.set_xlabel('Cosine Similarity', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Cosine Similarity Distribution (ECAPA-TDNN)', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('../results/cosine_similarity_dist.png', dpi=300)
    plt.show()